In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
from statsmodels.tsa.seasonal import seasonal_decompose
from template import API  # Assumes template.API.py is in the same directory or module

In [ ]:
# Fetch transaction count metric (or hash_rate/block_size)
df = API.fetch_bitcoin_metric("transaction_count")
df.tail()

In [ ]:
# Handle missing values (interpolate)
df["value"].interpolate(method="linear", inplace=True)

# Compute rolling mean and Z-score
df["rolling_mean"] = df["value"].rolling(window=10, min_periods=1).mean()
df["rolling_std"] = df["value"].rolling(window=10, min_periods=1).std()
df["z_score"] = (df["value"] - df["rolling_mean"]) / df["rolling_std"]

In [ ]:
# Decompose using statsmodels (requires sufficient points)
decomposition = seasonal_decompose(df["value"], model="additive", period=10)
df["trend"] = decomposition.trend
df["seasonal"] = decomposition.seasonal
df["residual"] = decomposition.resid

In [ ]:
fig = go.Figure()

# Actual metric
fig.add_trace(go.Scatter(x=df.index, y=df["value"], mode="lines", name="Value"))

# Rolling mean
fig.add_trace(go.Scatter(x=df.index, y=df["rolling_mean"], mode="lines", name="Rolling Mean"))

# Anomaly highlight
anomalies = df[df["z_score"].abs() > 2]
fig.add_trace(go.Scatter(
    x=anomalies.index, y=anomalies["value"],
    mode="markers", name="Anomalies", marker=dict(color="red", size=8)
))

fig.update_layout(title="Bitcoin Metric with Rolling Mean and Anomalies", xaxis_title="Date", yaxis_title="Metric Value")
fig.show()

In [ ]:
# Save as standalone HTML
pio.write_html(fig, file="bitcoin_metric_plot.html", auto_open=False)